# Analyzing experiments in Python

Data Scientist track, lesson DS13. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** size a test on a metric that is not a share, put a bootstrap interval on a mean, a ratio and a median, run the checks that need no outcome, and read a causal claim critically.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the app-events data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import math
import numbers
import os
import subprocess
import sys
import tempfile
import urllib.request

import numpy as np
import pandas as pd

try:
    import duckdb
except ImportError:  # Colab usually has DuckDB; install it when it does not
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb
from scipy import stats

SEED = 2026       # the seed of the whole Data Scientist track

EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.

RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps.items() if isinstance(traps, dict) else (traps or [])):
        if same(got, wrong, tol):
            print("Hint:", message)
    return False

## Data

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0, or from the repository's own copy of the same files when the network is not available.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
LOCAL = "../../datasets/app-events/v1/"      # the same files, when this notebook runs inside a clone of the repository


def fetch(name):
    """The path of one CSV file: downloaded from the tag, or the local copy."""
    try:
        path = os.path.join(tempfile.mkdtemp(), name + ".csv")
        urllib.request.urlretrieve(BASE + name + ".csv", path)
        return path
    except Exception:
        return LOCAL + name + ".csv"

con = duckdb.connect()                       # a new, empty database every time this cell runs
con.execute(f"CREATE TABLE users AS FROM read_csv('{fetch('users')}', types={{'signup_date': 'DATE'}})")
con.execute(f"CREATE TABLE events AS FROM read_csv('{fetch('events')}', types={{'event_ts': 'TIMESTAMP', 'received_at': 'TIMESTAMP'}})")


con = duckdb.connect()                       # a new, empty database every time this cell runs
con.execute(f"CREATE TABLE users AS FROM read_csv('{fetch('users')}', types={{'signup_date': 'DATE'}})")
con.execute(f"CREATE TABLE events AS FROM read_csv('{fetch('events')}', types={{'event_ts': 'TIMESTAMP', 'received_at': 'TIMESTAMP'}})")
con.execute(f"CREATE TABLE subscriptions AS FROM read_csv('{fetch('subscriptions')}', types={{'started_on': 'DATE', 'ended_on': 'DATE', 'updated_at': 'TIMESTAMP'}})")
con.execute(f"CREATE TABLE assign AS FROM read_csv('{fetch('experiment_assignments')}', types={{'assigned_at': 'TIMESTAMP'}})")

# Known answer from the dataset README: 21,841 people and 177,379 events.
assert con.execute("SELECT (SELECT COUNT(*) FROM users), (SELECT COUNT(*) FROM events)").fetchall()[0] == (21841, 177379)


# One row for each person in an experiment: who they are at signup, the arm, and what happened in the 14 days after they were put in it.
EXP_SQL = """
WITH first_version AS (          -- the plan a subscription started with: its earliest saved row
  SELECT subscription_id, user_id, plan, started_on
  FROM subscriptions
  QUALIFY ROW_NUMBER() OVER (PARTITION BY subscription_id ORDER BY updated_at) = 1
),
signup AS (
  SELECT user_id, platform FROM events WHERE event_name = 'signup_complete'
),
buys AS (                        -- each purchase in the 14 days after the person was put in the experiment, with its price
  SELECT a.user_id, a.experiment, e.event_ts,
         CASE f.plan WHEN 'plus_annual' THEN 59.99 ELSE 7.99 END AS price
  FROM assign AS a
  JOIN events AS e ON e.user_id = a.user_id AND e.event_name = 'purchase'
       AND e.event_ts >= a.assigned_at AND e.event_ts <= a.assigned_at + INTERVAL 14 DAY
  JOIN first_version AS f ON f.user_id = e.user_id AND f.started_on = CAST(e.event_ts AS DATE)
)
SELECT a.user_id, a.experiment, a.variant, a.assigned_at, a.converted,
       u.channel, s.platform, u.region,
       COALESCE(SUM(b.price), 0) AS revenue_14d,
       EXTRACT(epoch FROM (MIN(b.event_ts) - a.assigned_at)) / 3600 AS hours_to_buy
FROM assign AS a
JOIN users AS u ON u.user_id = a.user_id
JOIN signup AS s ON s.user_id = a.user_id
LEFT JOIN buys AS b ON b.user_id = a.user_id AND b.experiment = a.experiment
GROUP BY a.user_id, a.experiment, a.variant, a.assigned_at, a.converted, u.channel, s.platform, u.region
ORDER BY a.experiment, a.user_id
"""
exp = con.sql(EXP_SQL).df()
assert len(exp) == 15868 and sorted(exp["revenue_14d"].round(2).unique().tolist()) == [0.0, 7.99, 59.99]
print(f"{len(exp):,} people in three experiments:", exp.groupby("experiment").size().to_dict())

# The modeling table of the Data Scientist track: one row per person, built by the query of lesson DS3, "Building a modeling table in SQL".
# Run this cell as it is; DS3 explains it line by line.
MODELING_SQL = """WITH people AS (            -- one row per person: the spine of the table
  SELECT u.user_id, u.signup_date, u.channel, u.region, s.platform,
         s.event_ts AS signup_ts,
         s.event_ts + INTERVAL 7 DAY AS cutoff,       -- the prediction moment
         s.event_ts + INTERVAL 30 DAY AS horizon      -- the end of the label window
  FROM users AS u
  JOIN events AS s ON s.user_id = u.user_id AND s.event_name = 'signup_complete'
  WHERE u.signup_date BETWEEN DATE '2026-01-01' AND DATE '2026-05-31'
),
clean AS (                  -- retry copies removed: keep the first copy of each event
  SELECT *
  FROM events
  QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id, event_ts, event_name, platform ORDER BY received_at, event_id) = 1
),
known AS (                  -- what the app knew at the cutoff: it had happened and it had arrived
  SELECT p.user_id, c.event_name, c.event_ts, p.cutoff
  FROM people AS p
  JOIN clean AS c ON c.user_id = p.user_id
  WHERE c.event_ts <= p.cutoff AND c.received_at <= p.cutoff
),
features AS (
  SELECT user_id,
         COUNT(*) FILTER (WHERE event_name = 'session_start') AS sessions_7d,
         COUNT(*) FILTER (WHERE event_name = 'add_expense') AS expenses_7d,
         COUNT(*) FILTER (WHERE event_name IN ('session_start', 'add_expense') AND event_ts > cutoff - INTERVAL 3 DAY) AS recent_3d,
         COUNT(DISTINCT CAST(event_ts AS DATE)) FILTER (WHERE event_name IN ('session_start', 'add_expense')) AS active_days_7d,
         CAST(COUNT(*) FILTER (WHERE event_name = 'onboarding_complete') > 0 AS INTEGER) AS onboarded,
         CAST(COUNT(*) FILTER (WHERE event_name IN ('view_paywall', 'paywall_viewed')) > 0 AS INTEGER) AS saw_paywall,
         CAST(COUNT(*) FILTER (WHERE event_name = 'purchase') > 0 AS INTEGER) AS bought_plus
  FROM known
  GROUP BY user_id
),
label AS (                  -- from the other side of the cutoff: no visit and no expense in days 8 to 30
  SELECT p.user_id,
         CAST(NOT EXISTS (SELECT 1 FROM events AS e
                          WHERE e.user_id = p.user_id AND e.event_name IN ('session_start', 'add_expense')
                            AND e.event_ts > p.cutoff AND e.event_ts <= p.horizon) AS INTEGER) AS churned
  FROM people AS p
)
SELECT p.user_id, p.signup_date, p.channel, p.platform, p.region,
       f.sessions_7d, f.expenses_7d, f.recent_3d, f.active_days_7d, f.onboarded, f.saw_paywall, f.bought_plus,
       l.churned
FROM people AS p
JOIN features AS f ON f.user_id = p.user_id
JOIN label AS l ON l.user_id = p.user_id
ORDER BY p.user_id"""
table = con.sql(MODELING_SQL).df()
print(f"Data loaded: {len(table):,} people, {int(table['churned'].sum()):,} of them churned.")


## The three experiments

`exp` has one row for each person in an experiment. Run this cell as it is.

In [ ]:
pw = exp[exp["experiment"] == "paywall_v2"]              # the redesigned paywall: a real, modest lift
wt = exp[exp["experiment"] == "welcome_tips"]            # tips on the first screens: no effect
sf = exp[exp["experiment"] == "signup_flow_v2"]          # a new signup screen
print(len(pw), len(wt), len(sf))

## Unit 1: Size a test on revenue

**Predict first.** Revenue per person in 14 days is $0 for most people, $7.99 or $59.99 for a buyer. To catch a lift of $0.50 a person, about how many people does each arm need: about 700, about 7,000 or about 70,000?

In [ ]:
pilot = wt.loc[wt["variant"] == "control", "revenue_14d"].to_numpy()      # the control arm of the last test: a pilot for the next one
print(len(pilot), round(float(pilot.mean()), 3), round(float(pilot.std(ddof=1)), 3), round(float((pilot > 0).mean()), 3))


def n_per_arm(sd, delta, alpha=0.05, power=0.80):
    """People needed in each arm to catch a true difference of `delta` in a mean whose spread is `sd`."""
    z = stats.norm.ppf(1 - alpha / 2) + stats.norm.ppf(power)
    return int(np.ceil(2 * (sd * z / delta) ** 2))


rng = np.random.default_rng(SEED)


def power_at(n, delta, runs=2000):
    """Share of simulated tests with p < 0.05: draw n people per arm from the pilot; the variant's revenue is scaled up so its mean is `delta` higher."""
    scale = 1 + delta / pilot.mean()
    hits = 0
    for _ in range(runs):
        a, b = rng.choice(pilot, n), rng.choice(pilot, n) * scale
        hits += stats.ttest_ind(b, a, equal_var=False).pvalue < 0.05
    return hits / runs


n = n_per_arm(pilot.std(ddof=1), 0.50)
print(n, round(power_at(n, 0.50), 2), round(power_at(2700, 0.50), 2))        # the planned size, its simulated power, and the power of 2,700 per arm

## Unit 2: A bootstrap interval for any statistic

**Predict first.** What is the median revenue per person in each arm of `paywall_v2`: about $2.6, the average, $7.99, or $0?

In [ ]:
ctrl, var = pw[pw["variant"] == "control"], pw[pw["variant"] == "variant"]
rng = np.random.default_rng(SEED)


def bootstrap(stat, variant, control, reps=2000):
    """95% percentile interval of stat(variant rows, control rows): resample the people of each arm, then recompute the statistic."""
    draws = [stat(variant.iloc[rng.integers(0, len(variant), len(variant))], control.iloc[rng.integers(0, len(control), len(control))]) for _ in range(reps)]
    return np.percentile(draws, [2.5, 97.5])


stats_ = {
    "mean revenue, difference": lambda v, c: v["revenue_14d"].mean() - c["revenue_14d"].mean(),
    "conversion, relative lift": lambda v, c: v["converted"].mean() / c["converted"].mean() - 1,
    "median revenue, difference": lambda v, c: v["revenue_14d"].median() - c["revenue_14d"].median(),
    "median hours to buy, difference": lambda v, c: v["hours_to_buy"].median() - c["hours_to_buy"].median(),
}
print(round(float(ctrl["revenue_14d"].median()), 2), round(float(var["revenue_14d"].median()), 2))
for name, stat in stats_.items():
    low, high = bootstrap(stat, var, ctrl)
    print(f"{name}: {stat(var, ctrl):.3f} ({low:.3f} to {high:.3f})")
t = stats.ttest_ind(var["revenue_14d"], ctrl["revenue_14d"], equal_var=False).confidence_interval()
print(round(float(t.low), 3), round(float(t.high), 3))                          # the textbook interval for the mean difference

## Unit 3: Checks that need no outcome

**Predict first.** Which experiment fails which check? And in 1,000 random A/A splits of one arm, will about 1, about 5 or about 20 percent come out with p below 0.05?

In [ ]:
def two_prop_p(k0, n0, k1, n1):
    """Two-sided p of a z test for two shares, with the pooled standard error (as in lesson C14)."""
    p = (k0 + k1) / (n0 + n1)
    z = (k1 / n1 - k0 / n0) / np.sqrt(p * (1 - p) * (1 / n0 + 1 / n1))
    return 2 * stats.norm.sf(abs(z))


def sanity(g):
    """Checks that need no outcome: the split against 50/50, and each signup column against the arm."""
    n = g["variant"].value_counts()
    row = {"control": int(n["control"]), "variant": int(n["variant"]), "ratio": stats.chisquare([n["control"], n["variant"]]).pvalue}
    for col in ["channel", "platform", "region"]:
        row[col] = stats.chi2_contingency(pd.crosstab(g[col], g["variant"]))[1]
    return row


checks = pd.DataFrame({name: sanity(g) for name, g in exp.groupby("experiment")}).T
shown = checks.assign(**{c: checks[c].map(lambda v: f"{v:.4f}") for c in ["ratio", "channel", "platform", "region"]}, control=checks["control"].astype(int), variant=checks["variant"].astype(int))
print(shown.to_string())
rng = np.random.default_rng(SEED)
c = pw.loc[pw["variant"] == "control", "converted"].to_numpy()                   # 2,706 people who all got the same paywall
hits = 0
for _ in range(1000):                                                            # an A/A test: split them at random and run the readout
    i = rng.permutation(len(c))
    a, b = c[i[: len(c) // 2]], c[i[len(c) // 2:]]
    hits += two_prop_p(a.sum(), len(a), b.sum(), len(b)) < 0.05
print(round(hits / 1000, 2))

## Unit 4: Reading a causal claim

**Predict first.** People who finished onboarding churned 10.8 points less. Inside each channel, will the gap be about the same, about half, or about zero?

In [ ]:
by_channel = table.groupby(["channel", "onboarded"])["churned"].mean().unstack()
raw = table.groupby("onboarded")["churned"].mean()
weights = table["channel"].value_counts(normalize=True).reindex(by_channel.index)
adjusted = float(((by_channel[1] - by_channel[0]) * weights).sum())               # the gap inside each channel, averaged with the channel sizes
print(round(float(raw[1] - raw[0]), 3), round(adjusted, 3))
code_, o, y = table["channel"].astype("category").cat.codes.to_numpy(), table["onboarded"].to_numpy(), table["churned"].to_numpy()
rng = np.random.default_rng(SEED)
draws = []
for _ in range(2000):                                                            # resample the people, recompute the adjusted gap
    i = rng.integers(0, len(y), len(y))
    cell = code_[i] * 2 + o[i]
    rate = np.bincount(cell, weights=y[i], minlength=10) / np.bincount(cell, minlength=10)
    draws.append(((rate[1::2] - rate[0::2]) * np.bincount(code_[i], minlength=5) / len(i)).sum())
print(np.round(np.percentile(draws, [2.5, 97.5]), 3))
share = pd.crosstab(table["onboarded"], table["channel"], normalize="index")["paid_social"]
arms = pd.crosstab(pw["variant"], pw["channel"], normalize="index")["paid_social"]
print(share.round(3).to_dict(), arms.round(3).to_dict())                          # the paid-social share, by onboarding and by experiment arm

## Finish it

### Exercise 1: Finish it

**Task:** **A null test, with an interval.** `welcome_tips` changed nothing. Use `bootstrap` of unit 2 on the statistic `relative_revenue`, the relative change in mean revenue per person, for the variant and the control of `welcome_tips`. Put the list [low, high] of the interval in `answer_1`, each rounded to 2 decimals.

**Expected:** `answer_1` is a list of two numbers.

In [ ]:
wt_ctrl, wt_var = wt[wt["variant"] == "control"], wt[wt["variant"] == "variant"]
relative_revenue = lambda v, c: v["revenue_14d"].mean() / c["revenue_14d"].mean() - 1           # given: the relative change in mean revenue per person
# bootstrap this statistic for welcome_tips with bootstrap() of unit 2, and read the interval
answer_1 = None  # replace None with [low, high] of the interval, each rounded to 2 decimals

In [ ]:
_ = check("Exercise 1", "answer_1", [-0.28, 0.13], tol=3e-2)

<details>
<summary>Hint</summary>

`bootstrap(stat, variant, control)` takes the statistic first, then the variant rows, then the control rows. `wt_var` and `wt_ctrl` are ready.

</details>

<details>
<summary>Half-solution</summary>

```python
low, high = bootstrap(relative_revenue, ..., ...)
answer_1 = [round(float(low), 2), round(float(high), 2)]
```

</details>

<details>
<summary>Solution</summary>

```python
wt_ctrl, wt_var = wt[wt["variant"] == "control"], wt[wt["variant"] == "variant"]
relative_revenue = lambda v, c: v["revenue_14d"].mean() / c["revenue_14d"].mean() - 1
low, high = bootstrap(relative_revenue, wt_var, wt_ctrl)
answer_1 = [round(float(low), 2), round(float(high), 2)]
```

The relative change in mean revenue per person for `welcome_tips` is -0.10, and its interval, -0.28 to 0.13, holds 0: a test that changed nothing still cannot rule out a change of 20 percent either way at this size. The interval, not the point estimate, says what the data allow (seed 2026, 2,000 resamples).

</details>

## On your own

### Exercise 2: On your own

**Task:** **Where is the split off?** Before you start, say in one sentence what you expect: the ratio check of `signup_flow_v2` failed, so one platform's own split should fail and the others pass. Count the people of each arm inside each platform of `sf` and put the list of three p-values of the ratio check, for android, ios and web in that order, in `answer_2`, each rounded to 3 decimals. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is a list of three numbers.

In [ ]:
# write your code here

In [ ]:
_ = check("Exercise 2", "answer_2", [0.0, 0.014, 0.477], tol=3e-3)

<details>
<summary>Hint</summary>

`sf[sf["platform"] == level]["variant"].value_counts()` gives the two counts. Give them to `stats.chisquare` as [control, variant].

</details>

<details>
<summary>Half-solution</summary>

```python
for level in ["android", "ios", "web"]:
    g = sf[sf["platform"] == level]["variant"].value_counts()
    ...
```

</details>

<details>
<summary>Solution</summary>

```python
platform_p = []
for level in ["android", "ios", "web"]:
    g = sf[sf["platform"] == level]["variant"].value_counts()
    platform_p.append(round(float(stats.chisquare([g["control"], g["variant"]]).pvalue), 3))
answer_2 = platform_p
```

The split of android is far off (p below 0.0005, 1,020 against 656), iOS is at 0.014 (1,070 against 1,187) and web is at 0.477. A bar of 0.05 over three segments flags iOS too; the bar of 0.0005 that C14 used leaves only android, the platform the new screen crashes on. One segment that fails by a lot is the signal; one that fails by a little is expected now and then.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** **Spot the flaw.** A colleague sizes a test for +$0.40 a person with the spread of what *buyers* pay, "the spread of what a person pays". Put in `answer_3` the people needed in each arm when the spread is that of every person put in the test, as in unit 1.

**Expected:** `answer_3` is a whole number.

In [ ]:
buyers = pilot[pilot > 0]
sd_buyers = buyers.std(ddof=1)                                   # colleague: "the spread of what a person pays"
answer_3 = n_per_arm(sd_buyers, 0.40)                            # people per arm to catch +$0.40 a person at power 0.80
# size it with the spread of revenue for every person who is put in the test

In [ ]:
_ = check("Exercise 3", "answer_3", 11010, traps={50342: "That is the spread of the buyers only. Use the spread of every person put in the test."})

<details>
<summary>Hint</summary>

The test compares the mean over every person assigned, so the spread must be the spread over every person assigned: `pilot`, not `buyers`.

</details>

<details>
<summary>Half-solution</summary>

```python
answer_3 = n_per_arm(pilot....std(ddof=1), 0.40)
```

</details>

<details>
<summary>Solution</summary>

```python
answer_3 = n_per_arm(pilot.std(ddof=1), 0.40)
```

The spread of what buyers pay is $22.65, because every buyer pays $7.99 or $59.99, and it gives 50,342 per arm. The test compares the mean over everyone put in it, zeros included, whose spread is $10.59, and it gives 11,010 per arm: the colleague's plan is 4.6 times too large.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")